# Economy of Bits — Notebook 06: the bridge, and where it leads

Notebook 05 finished the measurement. The mechanism test came out as cleanly as this kind of
thing ever does:

| | capex/revenue 2016 → 2025 | asset lightness |
|---|---|---|
| **hyperscalers** | 0.102 → **0.277** | 0.805 → **0.584** |
| other tail firms | 0.067 → 0.064 | 0.705 → 0.711 |

This notebook adds no measurements. It turns those tables into tests, checks whether nine
firms can carry a finding, and builds **the bridge** — because the paper has two halves that
so far share a decade and nothing else: concentration rose, and the top became
capital-intensive. If capital spending predicts gains in value share they are one argument.

**It does not.** Cell 7 follows where that failure points, and it is the most interesting
cell in the notebook.

Upload `bits_inputs_v6.zip`. Runtime under 5 minutes — no bulk downloads.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 05 finished the measurement. The mechanism test came out as cleanly
# as this kind of thing ever does:
#
#                        capex / revenue        asset lightness
#                        2016 -> 2025           2016 -> 2025
#   hyperscalers         0.102 -> 0.277         0.805 -> 0.584
#   other tail firms     0.067 -> 0.064         0.705 -> 0.711
#
# The capital turn is entirely in the four firms that OPERATE compute. Nvidia,
# which designs chips and owns no fabs, went 0.036 -> 0.028. Apple, which
# outsources assembly, went 0.059 -> 0.031 -- falling. And the leases objection
# is answered three ways: capex is a cash-flow measure ASC 842 cannot touch,
# the two measures correlate at -0.77, the decline survives on plain-tag
# firm-years at -0.0073 (p < 0.0001), and the 2019 step is POSITIVE, because
# right-of-use assets inflate the denominator -- leases mask the decline rather
# than manufacture it.
#
# This notebook does not add measurements. It does four things the paper needs
# before a word of it can be written:
#
#   1. Turns the two-column tables into tests with coefficients and p-values.
#   2. Checks the tail definition, which currently rests on nine firms.
#   3. Builds the BRIDGE. The paper has two halves -- concentration rose, and
#      the top became capital-intensive -- and nothing yet connects them. If
#      capital spending predicts gains in value share, they are one argument.
#      If it does not, they are two observations that happen to share a decade.
#   4. Fixes what is still broken: Berkshire has no PP&E in ten years, and the
#      depreciation series is not credible (balanced digital firms at
#      0.077 -> 0.286, which is a tagging artefact, not an economy).
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd, requests
import statsmodels.api as sm
from scipy import stats as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 160)

RUN_TAG = "bits_06_bridge"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, CACHE, FIGS = ROOT / "outputs", ROOT / "cache", ROOT / "outputs" / "figures"
for d in (OUTD, CACHE, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

SEC_HEADERS = {"User-Agent": "Boston University Metropolitan College research "
                             "(spattna1@bu.edu)", "Accept-Encoding": "gzip, deflate"}

def sec_get(url, tries=3, timeout=60):
    for a in range(tries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=SEC_HEADERS, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                return None
            if r.status_code in (403, 429):
                time.sleep(5 * (a + 1)); continue
            return None
        except Exception:
            if a == tries - 1:
                return None
            time.sleep(2 * (a + 1))
    return None

PAPER_YEARS = list(range(2016, 2026))
HYPERSCALERS = {"MSFT", "GOOGL", "GOOG", "AMZN", "META"}
print(f"writing to: {ROOT}")

## 2 · Inputs, and two loose ends

In [ ]:
# =============================================================================
# CELL 2 - Inputs, plus the two loose ends
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v6.zip"), recursive=True)):
        dest = CACHE / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs6", BASE, CACHE / "inputs"):
        if (pathlib.Path(d) / "firm_measures_v3.csv").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v6.zip not found.")

F = pd.read_csv(SRC / "firm_measures_v3.csv"); F["year"] = F.year.astype(int)
BI = pd.read_csv(SRC / "bit_intensity_v3.csv")
facts = pd.read_csv(SRC / "edgar_facts_v3.csv.gz")
coding = pd.read_csv(SRC / "platform_coding_final.csv")
F = F.merge(BI[["year", "firm_id", "BI", "BI_sector_neutral"]],
            on=["year", "firm_id"], how="left")
F["firm_id"] = F.firm_id.astype(str)
print(f"{len(F):,} firm-years, {F.firm_id.nunique()} firms")

# --- loose end 1: Berkshire has no PP&E ---------------------------------------
# Berkshire reports property under insurance and railroad segment tags rather
# than the standard one. Try the alternatives before excluding it, because it
# sits in the tail group and an unexplained hole there is not acceptable.
BRK_TAGS = ["PropertyPlantAndEquipmentNet",
            "PropertyPlantAndEquipmentAndFinanceLeaseRightOfUseAssetAfterAccumulatedDepreciationAndAmortization",
            "PropertySubjectToOrAvailableForOperatingLeaseNet",
            "PropertyPlantAndEquipmentOtherNet", "RailroadsAndEquipmentNet"]
brk = F[F.ticker == "BRK-B"]
if len(brk) and brk.asset_light.isna().all():
    cik = int(brk.cik.dropna().iloc[0]) if brk.cik.notna().any() else 1067983
    js = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json")
    gaap = (js or {}).get("facts", {}).get("us-gaap", {})
    found = {t: len(gaap[t].get("units", {}).get("USD", [])) for t in BRK_TAGS if t in gaap}
    print(f"\nBerkshire (CIK {cik}) property tags present: {found or 'none of the candidates'}")
    if not found:
        pp = [k for k in gaap if "PropertyPlantAndEquipment" in k][:8]
        print(f"  tags containing PropertyPlantAndEquipment: {pp}")
    print("  If none carries an annual USD series, Berkshire is excluded from the")
    print("  capital measures and the paper says so -- one firm, named, not a")
    print("  silent gap.")

# --- loose end 2: depreciation is not credible --------------------------------
# Balanced digital firms at 0.077 -> 0.286 is not an economy, it is a tag
# problem: DepreciationDepletionAndAmortization and its variants are reported
# on different bases by different filers.
print("\ndepreciation intensity, distribution by year:")
display(F.groupby("year").dep_intensity.describe()[["count", "mean", "50%", "max"]].round(3))
tags = facts[facts.field == "depreciation"].tag.value_counts()
print(f"\ntags behind it: {dict(tags)}")
print("A median far below the mean, or a maximum near 1, means the series mixes")
print("bases. Depreciation is DROPPED from the paper unless this looks clean.")
DROP_DEPRECIATION = True

## 3 · Can nine firms carry it?

In [ ]:
# =============================================================================
# CELL 3 - The tail definition, and whether nine firms can carry a paper
#
# "Ever in the top 1%" gives nine firms, four of which are the hyperscalers.
# A result resting on four firms is not necessarily wrong -- the top 1% of the
# S&P 500 IS four or five firms, that is what the first paper measured -- but
# the paper has to show the finding is not an artefact of where the line was
# drawn. So: three definitions, and the same trend estimated under each.
# =============================================================================
def make_groups(df, q):
    d = df.copy()
    intop = d.groupby("year").cap_share.transform(lambda s: s >= s.quantile(1 - q))
    tail = set(d.loc[intop, "firm_id"].unique())
    d["grp"] = np.where(d.firm_id.isin(tail), "tail",
                        np.where(d.gics_digital, "digital", "rest"))
    return d, tail

def trend(s):
    s = s.dropna()
    if len(s) < 5:
        return np.nan, np.nan
    X = sm.add_constant(s.index.values - np.mean(s.index.values))
    f = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    return float(f.params[1]), float(f.pvalues[1])

rows = []
for q, lab in [(0.01, "top 1%"), (0.05, "top 5%"), (0.10, "top 10%")]:
    d, tail = make_groups(F, q)
    for var in ["capex_intensity", "asset_light"]:
        piv = d.pivot_table(index="year", columns="grp", values=var)
        for g in piv.columns:
            b, p = trend(piv[g])
            rows.append({"definition": lab, "n_tail_firms": len(tail), "variable": var,
                         "group": g, "first": round(float(piv[g].dropna().iloc[0]), 4),
                         "last": round(float(piv[g].dropna().iloc[-1]), 4),
                         "slope": round(b, 5) if b == b else np.nan,
                         "p": round(p, 4) if p == p else np.nan})
robust = pd.DataFrame(rows)
print("the same trend under three definitions of 'the tail':")
display(robust)
savetable(robust, "tail_definition_robustness")
print("\nIf the tail slope keeps its sign and significance as the group widens")
print("from 9 firms to roughly 80, the finding is about the top of the")
print("distribution and not about four companies.")

## 4 · The divergence, as a coefficient

In [ ]:
# =============================================================================
# CELL 4 - The divergence, as a coefficient rather than two columns
#
# Notebook 05 showed hyperscalers and other tail firms moving apart in a table.
# A table is not a test. This is a difference-in-differences with firm fixed
# effects, so each firm is compared with itself over time and the coefficient
# on hyperscaler x trend is the divergence:
#
#   y_it = a_i + b*t + c*(hyperscaler_i x t) + e_it
#
# Firm fixed effects absorb the fact that hyperscalers are simply different
# firms; only the DIFFERENCE IN TREND identifies c. Standard errors are
# clustered by firm, because ten observations of Microsoft are one firm.
# =============================================================================
def did(df, var, treat_col, label, sample=""):
    d = df.dropna(subset=[var]).copy()
    d["t"] = d.year - 2016
    fe = pd.get_dummies(d.firm_id, prefix="f", drop_first=True).astype(float)
    X = pd.concat([fe.reset_index(drop=True),
                   pd.DataFrame({"t": d.t.values,
                                 "treat_x_t": (d[treat_col].astype(float) * d.t).values})],
                  axis=1)
    X = sm.add_constant(X)
    f = sm.OLS(d[var].values, X.values.astype(float)).fit(
        cov_type="cluster", cov_kwds={"groups": d.firm_id.values})
    i = list(X.columns).index("treat_x_t")
    j = list(X.columns).index("t")
    return {"variable": var, "contrast": label, "sample": sample, "n": len(d),
            "n_firms": int(d.firm_id.nunique()),
            "base_trend": round(float(f.params[j]), 5),
            "extra_trend": round(float(f.params[i]), 5),
            "se": round(float(f.bse[i]), 5),
            "t_stat": round(float(f.tvalues[i]), 2),
            "p": round(float(f.pvalues[i]), 4)}

F["hyperscaler"] = F.ticker.isin(HYPERSCALERS)
d_tail, tail_firms = make_groups(F, 0.01)
F["in_tail"] = F.firm_id.isin(tail_firms)

rows = []
for var in ["capex_intensity", "asset_light", "gross_margin"]:
    rows.append(did(F[F.in_tail], var, "hyperscaler",
                    "hyperscaler vs other tail firms", "tail only"))
    rows.append(did(F, var, "hyperscaler", "hyperscaler vs everyone else", "all firms"))
    rows.append(did(F, var, "in_tail", "tail vs everyone else", "all firms"))
    rows.append(did(F[F.gics_digital], var, "in_tail",
                    "tail vs other digital firms", "digital only"))
didres = pd.DataFrame(rows)
print("difference-in-differences, firm fixed effects, SEs clustered by firm:\n")
display(didres)
savetable(didres, "did_divergence")

print("\nThe row that carries the mechanism is 'hyperscaler vs other tail firms'")
print("on capex intensity. Both groups are at the very top of the index, so")
print("size, market power and index membership are held constant; what differs")
print("is whether the firm operates compute. If the extra trend is positive and")
print("significant there, the story is about compute and not about bigness.")

## 5 · The bridge — did capital buy share?

Lagged capex, firm and year fixed effects, clustered errors. A firm is compared with its own
other years, so this is not "big firms spend and are big".

In [ ]:
# =============================================================================
# CELL 5 - THE BRIDGE
#
# The paper currently has two halves that share a decade and nothing else:
#
#   A. concentration rose -- Pareto alpha 1.467 -> 1.141, top 1% share 11% -> 30%
#   B. the top of the index became capital-intensive
#
# Unless something connects them, the paper reports two facts and calls it an
# argument. The connection worth testing is the obvious one: did the firms that
# spent on capital GAIN VALUE SHARE?
#
#   dlog(cap_share)_it = a_i + d_t + b*capex_intensity_i,t-1 + controls + e_it
#
# Lagged capital spending, firm and year fixed effects, clustered errors. Firm
# effects mean a firm is compared with its own other years, so this is not
# "big firms spend and are big"; year effects remove the market. A positive b
# says spending on capital preceded gains in share.
#
# Read the sign carefully. This is a predictive association inside a decade with
# one enormous common shock, not an identified causal effect, and the paper must
# say so in those words.
# =============================================================================
G = F.sort_values(["firm_id", "year"]).copy()
G["lag_share"] = G.groupby("firm_id").cap_share.shift(1)
G["lag_year"] = G.groupby("firm_id").year.shift(1)
G["lag_capex"] = G.groupby("firm_id").capex_intensity.shift(1)
G["lag_BI"] = G.groupby("firm_id").BI.shift(1)
G["lag_gm"] = G.groupby("firm_id").gross_margin.shift(1)
G = G[(G.year - G.lag_year) == 1]
G["d_log_share"] = np.log(G.cap_share / G.lag_share)
G = G.replace([np.inf, -np.inf], np.nan)
print(f"{len(G):,} firm-year changes, {G.firm_id.nunique()} firms")

def bridge(df, regs, label):
    d = df.dropna(subset=["d_log_share"] + regs).copy()
    fe = pd.get_dummies(d.firm_id, prefix="f", drop_first=True).astype(float)
    yd = pd.get_dummies(d.year, prefix="y", drop_first=True).astype(float)
    X = pd.concat([fe.reset_index(drop=True), yd.reset_index(drop=True),
                   d[regs].reset_index(drop=True)], axis=1)
    X = sm.add_constant(X)
    f = sm.OLS(d.d_log_share.values, X.values.astype(float)).fit(
        cov_type="cluster", cov_kwds={"groups": d.firm_id.values})
    out = {"model": label, "n": len(d), "n_firms": int(d.firm_id.nunique())}
    for r in regs:
        k = list(X.columns).index(r)
        out[f"{r}_coef"] = round(float(f.params[k]), 4)
        out[f"{r}_p"] = round(float(f.pvalues[k]), 4)
    return out

rows = [bridge(G, ["lag_capex"], "capex only"),
        bridge(G, ["lag_capex", "lag_BI"], "capex + bit intensity"),
        bridge(G, ["lag_capex", "lag_gm"], "capex + gross margin")]
G["lag_capex_x_digital"] = G.lag_capex * G.gics_digital.astype(float)
rows.append(bridge(G, ["lag_capex", "lag_capex_x_digital"],
                   "capex, and capex interacted with digital"))
br = pd.DataFrame(rows)
print("\ndoes last year's capital spending predict this year's gain in value share?")
print("(firm and year fixed effects, SEs clustered by firm)\n")
display(br)
savetable(br, "bridge_capex_to_share")

# The same question asked the other way, which is the version a sceptic will
# want: over the whole decade, is the CHANGE in a firm's share related to its
# AVERAGE capital intensity? Cross-sectional, no fixed effects to hide behind.
agg = (F.dropna(subset=["capex_intensity"]).groupby("firm_id")
       .agg(firm=("firm", "first"), mean_capex=("capex_intensity", "mean"),
            years=("year", "nunique"),
            share_first=("cap_share", lambda s: s.iloc[0]),
            share_last=("cap_share", lambda s: s.iloc[-1]),
            digital=("gics_digital", "max"), hyper=("hyperscaler", "max")))
agg = agg[agg.years >= 8].copy()
agg["d_share_log"] = np.log(agg.share_last / agg.share_first)
X = sm.add_constant(pd.DataFrame({"mean_capex": agg.mean_capex.values,
                                  "digital": agg.digital.astype(float).values}))
f = sm.OLS(agg.d_share_log.values, X.values.astype(float)).fit(cov_type="HC1")
print(f"\ncross-section, {len(agg)} firms with at least 8 years:")
print(f"  mean capex intensity  coef {float(f.params[1]):+.4f}  p = {float(f.pvalues[1]):.4f}")
print(f"  digital               coef {float(f.params[2]):+.4f}  p = {float(f.pvalues[2]):.4f}")
print(f"  R-squared {f.rsquared:.3f}")
print("\ntop 12 firms by decade share gain, with their capital intensity:")
display(agg.sort_values("d_share_log", ascending=False).head(12)
        [["firm", "mean_capex", "d_share_log", "digital", "hyper"]].round(3))
savetable(agg.reset_index(), "share_gain_cross_section")

## 6 · The last two years, handled carefully

In [ ]:
# =============================================================================
# CELL 6 - The last two years, handled carefully
#
# Hyperscaler capital intensity goes 0.146 (2023) -> 0.184 (2024) -> 0.277
# (2025). That is the AI build-out and it is the most topical thing in the data,
# which is exactly why it needs the most discipline. Two years is not a trend,
# and a paper that leans on 2024-25 is a paper about a moment.
#
# What can honestly be said: the level, the break, and how much of the decade's
# movement sits in the last two years. Reporting the decade with and without
# them is the test of whether the finding depends on the moment.
# =============================================================================
sub = F[F.in_tail].copy()
sub["kind"] = np.where(sub.hyperscaler, "hyperscaler", "other tail firm")
piv = sub.pivot_table(index="year", columns="kind", values="capex_intensity")
print("capital intensity, and how much of the change is 2024-25:")
display(piv.round(4))
for k in piv.columns:
    s = piv[k].dropna()
    full = s.iloc[-1] - s.iloc[0]
    to23 = s.loc[2023] - s.iloc[0] if 2023 in s.index else np.nan
    print(f"  {k:<18} 2016-2025 {full:+.4f}, of which through 2023 {to23:+.4f} "
          f"({100*to23/full:.0f}%)" if full else "")

print("\nthe same trends estimated on 2016-2023 only:")
rows = []
for var in ["capex_intensity", "asset_light"]:
    for kind, s2 in sub.groupby("kind"):
        for end, lab in [(2023, "2016-2023"), (2025, "2016-2025")]:
            p = s2[s2.year <= end].groupby("year")[var].mean().dropna()
            if len(p) < 5:
                continue
            X = sm.add_constant(p.index.values - np.mean(p.index.values))
            f = sm.OLS(p.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})
            rows.append({"variable": var, "group": kind, "window": lab,
                         "slope": round(float(f.params[1]), 5),
                         "p": round(float(f.pvalues[1]), 4)})
excl = pd.DataFrame(rows)
display(excl)
savetable(excl, "with_and_without_2024_25")
print("\nIf the hyperscaler slope is already significant through 2023, the")
print("finding is a decade-long reallocation that AI accelerated. If it only")
print("appears once 2024-25 is included, the paper is describing two years and")
print("should say so in the title as well as the text.")

## 7 · The bridge failed — follow where it points

The firms that gained the most value share are the ones that *sell* compute, not the ones
that buy it. This cell is exploratory and labelled as such.

In [ ]:
# =============================================================================
# CELL 7 - The bridge failed. Follow where it points.
#
# Cell 5 asked whether capital spending predicted gains in value share. It does
# not: the panel coefficient is -0.115 (p = 0.067) and the cross-sectional one
# -0.435 (p = 0.19). Both negative. What predicts share gain is being digital
# (+0.560, p = 0.0001) -- not spending on capital.
#
# And look at who actually gained the most share over the decade:
#
#   AMD +2.54, Broadcom +1.90, Micron +1.40, KLA +1.37, Lam Research +1.34,
#   Arista +1.27, Cadence +0.97, Synopsys +0.95
#
# Semiconductors, semiconductor equipment and EDA. Their mean capital intensity
# is around 0.03. These are the firms that SELL compute. The hyperscalers, who
# BUY it, ran capital intensity from 0.10 to 0.28 and did not lead the share
# gains. Nvidia is the limiting case: fabless, capex 0.028, and it became the
# largest firm in the index.
#
# So the economy of bits did not end. It MOVED. A division of labour opened
# inside the digital economy between firms that operate compute and firms that
# design it, and the non-rival, IP-heavy, asset-light economics that chapter 6
# describes now sit with the designers -- who captured the value.
#
# HONESTY ABOUT WHAT THIS IS. The layer classification below was written AFTER
# seeing the cross-section above. It is exploratory, it is labelled as such, and
# the paper must present it as a hypothesis this data generated rather than one
# it tested. The confirmation has to come from out-of-sample evidence: other
# indices, or the years after this sample.
# =============================================================================
LAYER = {}
# Operate compute: run hyperscale data centres, buy the chips.
for t in ["MSFT", "GOOGL", "GOOG", "AMZN", "META", "ORCL"]:
    LAYER[t] = "operates compute"
# Design compute: fabless chip designers and the EDA firms whose product is IP.
# Nobody here owns a fab.
for t in ["NVDA", "AMD", "AVGO", "QCOM", "MRVL", "SNPS", "CDNS", "ANET", "ADI",
          "MPWR", "SWKS", "QRVO", "NXPI"]:
    LAYER[t] = "designs compute"
# Fabricate compute: own the fabs, or sell the machines that fill them. Rival,
# capital-intensive, and the physical bottleneck.
for t in ["INTC", "MU", "TXN", "AMAT", "LRCX", "KLAC", "TER", "ON", "GFS", "WDC", "STX"]:
    LAYER[t] = "fabricates compute"

F["layer"] = F.ticker.map(LAYER)
# Assign into the missing rows rather than calling fillna with an ndarray:
# pandas rejects an array as a fill value, and a boolean mask is clearer anyway.
_unmapped = F.layer.isna()
F.loc[_unmapped, "layer"] = np.where(
    F.loc[_unmapped, "gics_digital"].astype(bool), "other digital", "not digital")
print("firms per layer:")
print(F.groupby("layer").firm.nunique().to_string())
print("\nfirms in each compute layer:")
for L in ["operates compute", "designs compute", "fabricates compute"]:
    print(f"  {L:<20} {sorted(F.loc[F.layer == L, 'firm'].unique())}")

print("\ncapital intensity and asset lightness by layer:")
for var in ["capex_intensity", "asset_light"]:
    piv = F.pivot_table(index="year", columns="layer", values=var)
    print(f"\n{var}:")
    display(piv.round(4))
    rows = []
    for L in piv.columns:
        b, p = trend(piv[L])
        s = piv[L].dropna()
        rows.append({"variable": var, "layer": L,
                     "first": round(float(s.iloc[0]), 4), "last": round(float(s.iloc[-1]), 4),
                     "slope": round(b, 5), "p": round(p, 4)})
    display(pd.DataFrame(rows))
    savetable(pd.DataFrame(rows), f"layer_trend_{var}")

# --- who captured the value? --------------------------------------------------
lay = (F.dropna(subset=["cap_share"]).groupby(["firm_id", "layer"], as_index=False)
       .agg(firm=("firm", "first"), years=("year", "nunique"),
            first_share=("cap_share", lambda s: s.iloc[0]),
            last_share=("cap_share", lambda s: s.iloc[-1]),
            mean_capex=("capex_intensity", "mean")))
lay = lay[lay.years >= 8].copy()
lay["d_log_share"] = np.log(lay.last_share / lay.first_share)
summ = (lay.groupby("layer")
        .agg(n_firms=("firm_id", "nunique"),
             median_share_gain=("d_log_share", "median"),
             mean_share_gain=("d_log_share", "mean"),
             mean_capex=("mean_capex", "mean"),
             total_share_2016=("first_share", "sum"),
             total_share_2025=("last_share", "sum"))
        .sort_values("median_share_gain", ascending=False))
summ["share_point_gain"] = (100 * (summ.total_share_2025 - summ.total_share_2016)).round(2)
print("\nWHO CAPTURED THE VALUE, by layer:")
display(summ.round(3))
savetable(summ.reset_index(), "layer_value_capture")

base = lay[lay.layer == "not digital"].d_log_share
print("\nshare gain against non-digital firms (Welch t-test):")
rows = []
for L in ["designs compute", "operates compute", "fabricates compute", "other digital"]:
    x = lay.loc[lay.layer == L, "d_log_share"].dropna()
    if len(x) < 3:
        continue
    t_, p_ = st.ttest_ind(x, base.dropna(), equal_var=False)
    rows.append({"layer": L, "n": len(x), "median_gain": round(float(x.median()), 3),
                 "vs_non_digital_t": round(float(t_), 2), "p": round(float(p_), 4)})
lt = pd.DataFrame(rows)
display(lt)
savetable(lt, "layer_share_gain_tests")
print("\nThe prediction this generates, and which the paper should state as a")
print("prediction rather than a result: firms that DESIGN compute stay")
print("asset-light and capture value; firms that OPERATE it become")
print("capital-intensive; firms that FABRICATE it were always capital-intensive.")
print("The economics of bits did not disappear from the economy. They moved up")
print("one layer, to the firms whose product is a design rather than a machine.")

## 8 · Paper tables and figures

In [ ]:
# =============================================================================
# CELL 7 - The paper's tables and figures
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY = "#1f3b57", "#c8702a", "#7a7a7a"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
ax = axes[0]
for k, c in [("hyperscaler", INK), ("other tail firm", ACC)]:
    if k in piv.columns:
        ax.plot(piv.index, piv[k], "o-", color=c, lw=2, ms=4, label=k)
rest = F[~F.in_tail].groupby("year").capex_intensity.mean()
ax.plot(rest.index, rest.values, "o-", color=GREY, lw=1.6, ms=3, label="rest of index")
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("Who is buying the capital"); ax.legend(fontsize=8)

ax = axes[1]
pa = sub.pivot_table(index="year", columns="kind", values="asset_light")
for k, c in [("hyperscaler", INK), ("other tail firm", ACC)]:
    if k in pa.columns:
        ax.plot(pa.index, pa[k], "o-", color=c, lw=2, ms=4, label=k)
ra = F[~F.in_tail].groupby("year").asset_light.mean()
ax.plot(ra.index, ra.values, "o-", color=GREY, lw=1.6, ms=3, label="rest of index")
ax.set_xlabel("year"); ax.set_ylabel("1 - PP&E / assets")
ax.set_title("And it shows on the balance sheet"); ax.legend(fontsize=8)

ax = axes[2]
a = agg.dropna(subset=["mean_capex", "d_share_log"])
ax.scatter(a.mean_capex, a.d_share_log, s=14, alpha=0.45, color=GREY,
           edgecolors="none", label="all firms")
h = a[a.hyper.astype(bool)]
ax.scatter(h.mean_capex, h.d_share_log, s=45, color=INK, label="hyperscalers")
for _, r in h.iterrows():
    ax.annotate(str(r.firm), (r.mean_capex, r.d_share_log), fontsize=7,
                xytext=(4, 3), textcoords="offset points", color=INK)
ax.axhline(0, color=GREY, lw=1, ls=":")
ax.set_xlabel("mean capex / revenue, 2016-2025")
ax.set_ylabel("log change in share of index value")
ax.set_title("The bridge"); ax.legend(fontsize=8)
fig.suptitle("Notebook 06: capital, and whether it bought share",
             fontsize=10, color=GREY, y=1.02)
savefig(fig, "B1_bridge")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
rows = []
for p in files:
    row = {"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size / 1024, 1)}
    if p.suffix == ".parquet":
        try:
            row["rows"] = len(pd.read_parquet(p))
        except Exception:
            row["rows"] = None
    rows.append(row)
manifest = pd.DataFrame(rows); manifest.to_csv(OUTD / "MANIFEST.csv", index=False)
display(manifest)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "tail_firms": sorted(F.loc[F.in_tail, "firm"].unique().tolist()),
     "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  did_divergence.csv           the mechanism, as coefficients")
print("  bridge_capex_to_share.csv    whether capital bought share")
print("  share_gain_cross_section.csv the same question without fixed effects")
print("  tail_definition_robustness.csv   whether nine firms carry it")
print("  with_and_without_2024_25.csv     whether it survives dropping the AI years")
print()
print("If the divergence is significant, the bridge coefficient is positive, and")
print("the trend holds through 2023, the empirical work is DONE and the next")
print("message from me is a draft rather than a notebook.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")